In [1]:
# Minimal setup
from pathlib import Path
import re
import random

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "working_data" / "nhamcs_data_2018_22.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data path exists: {DATA_PATH.exists()}")

Project root: /home/gaurav/python/kaggle/triage
Data path exists: True


In [2]:
# Load CSV and keep only complaint text + acuity target
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.astype(str).str.strip()

TEXT_COL = "chief_complaint_text"
TARGET_CANDIDATES = ["target_triage_acuity", "triage_acuity", "target"]
target_col = next((c for c in TARGET_CANDIDATES if c in df.columns), None)

if target_col is None:
    raise KeyError(
        "Target column not found. "
        f"Tried {TARGET_CANDIDATES}. First columns: {list(df.columns[:15])}"
    )

data = (
    df[[TEXT_COL, target_col]]
    .rename(columns={target_col: "target_triage_acuity"})
    .dropna(subset=[TEXT_COL, "target_triage_acuity"])
    .copy()
)
data["target_triage_acuity"] = data["target_triage_acuity"].astype(int)

def map_esi_to_group(val: int):
    if val in (1, 2):
        return 0
    if val == 3:
        return 1
    if val in (4, 5):
        return 2
    return np.nan

data["target_triage_acuity"] = data["target_triage_acuity"].map(map_esi_to_group)
data = data.dropna(subset=["target_triage_acuity"]).copy()
data["target_triage_acuity"] = data["target_triage_acuity"].astype(int)

display(data.head(10))
print(f"Rows used: {len(data):,}")
display(data["target_triage_acuity"].value_counts().sort_index().rename("count").to_frame())

,chief_complaint_text,target_triage_acuity
0,fever. throat soreness,2
1,"injury, other and unspecified, of foo.... foot...",2
2,fever. cough,2
3,"abdominal pain, cramps, spasms. chest pain. po...",2
4,"injury, other and unspecified, of foo...",2
5,fever,2
6,cough. nasal congestion,1
7,cough,2
8,"injury, other and unspecified of head...",2
9,skin rash,1


Rows used: 58,124


,count
target_triage_acuity,
0,9443
1,29568
2,19113


In [5]:
# Expand common short forms in chief complaint text
ABBR_MAP = {
    "sob": "shortness of breath",
    "cp": "chest pain",
    "abd": "abdominal",
    "ha": "headache",
    "n/v": "nausea and vomiting",
    "s/p": "status post",
    "w/": "with",
    "w/o": "without",
    "fx": "fracture",
    "lac": "laceration",
    "loc": "loss of consciousness",
    "mva": "motor vehicle accident",
    "uti": "urinary tract infection",
    "uri": "upper respiratory infection",
    "usp": "unspecified",
    "foo": "foot",
    "...": "",
    "oth": "other",
    
}

def expand_text(text: str) -> str:
    txt = str(text).lower().strip()
    txt = re.sub(r"\s+", " ", txt)

    # Replace slash abbreviations first.
    for k in ["n/v", "s/p", "w/o", "w/"]:
        txt = re.sub(rf"(?<!\w){re.escape(k)}(?!\w)", ABBR_MAP[k], txt)

    for abbr, full in ABBR_MAP.items():
        if "/" in abbr:
            continue
        txt = re.sub(rf"(?<![a-z0-9]){re.escape(abbr)}(?![a-z0-9])", full, txt)

    txt = re.sub(r"[^a-z0-9\s\-\.,]", " ", txt)
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

data["text_clean"] = data["chief_complaint_text"].map(expand_text)
display(data[["chief_complaint_text", "text_clean", "target_triage_acuity"]].head(10))

,chief_complaint_text,text_clean,target_triage_acuity
0,fever. throat soreness,fever. throat soreness,2
1,"injury, other and unspecified, of foo.... foot...","injury, other and unspecified, of foot. foot a...",2
2,fever. cough,fever. cough,2
3,"abdominal pain, cramps, spasms. chest pain. po...","abdominal pain, cramps, spasms. chest pain. po...",2
4,"injury, other and unspecified, of foo...","injury, other and unspecified, of foot...",2
5,fever,fever,2
6,cough. nasal congestion,cough. nasal congestion,1
7,cough,cough,2
8,"injury, other and unspecified of head...","injury, other and unspecified of head...",2
9,skin rash,skin rash,1


In [3]:
# DistilBERT (uncased) + CORN ordinal head with threshold calibration
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report
from scipy.optimize import minimize
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "nlpie/distil-clinicalbert"
NUM_CLASSES = 3
MAX_LEN = 100
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 16
LR = 2e-5
WEIGHT_DECAY = 0.1
EPOCHS = 2

print(f"Device: {DEVICE}")

class OrdinalTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=96):
        self.texts = texts.astype(str).tolist()
        self.labels = labels.astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["label"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class OrdinalBert(nn.Module):
    def __init__(self, model_name: str, num_classes: int = 5, dropout: float = 0.5):
        super().__init__()
        try:
            self.backbone = AutoModel.from_pretrained(
                model_name,
                use_safetensors=True,
            )
        except Exception as e:
            raise RuntimeError(
                "Failed to load backbone with safetensors. "
                "This environment blocks .bin checkpoint loading when torch<2.6 "
                "(CVE-2025-32434 mitigation). "
                "Use a model that provides safetensors or upgrade torch to >=2.6."
            ) from e
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.ordinal_head = nn.Linear(hidden, num_classes - 1)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_state = out.last_hidden_state[:, 0]
        return self.ordinal_head(self.dropout(cls_state))

def corn_loss(logits: torch.Tensor, y_idx: torch.Tensor, num_classes: int = 5) -> torch.Tensor:
    # CORN trains each threshold conditionally: P(y > k | y >= k).
    total_loss = torch.tensor(0.0, device=logits.device)
    total_count = 0

    for k in range(num_classes - 1):
        if k == 0:
            mask = torch.ones_like(y_idx, dtype=torch.bool)
        else:
            mask = y_idx >= k

        if mask.sum() == 0:
            continue

        target = (y_idx[mask] > k).float()
        loss_k = F.binary_cross_entropy_with_logits(logits[mask, k], target, reduction="sum")
        total_loss = total_loss + loss_k
        total_count += int(mask.sum().item())

    return total_loss / max(1, total_count)

def corn_cumulative_probs(logits: torch.Tensor) -> torch.Tensor:
    cond_probs = torch.sigmoid(logits)
    return torch.cumprod(cond_probs, dim=1)

def predict_from_logits(logits: torch.Tensor, thresholds=None) -> torch.Tensor:
    cum_probs = corn_cumulative_probs(logits)
    if thresholds is None:
        thresholds = torch.full((NUM_CLASSES - 1,), 0.5, device=cum_probs.device, dtype=cum_probs.dtype)
    else:
        thresholds = torch.as_tensor(thresholds, device=cum_probs.device, dtype=cum_probs.dtype)
    return (cum_probs > thresholds).sum(dim=1).long()

def find_optimal_thresholds(logits: torch.Tensor, y_true):
    probs = corn_cumulative_probs(logits).detach().cpu().numpy()
    y_true = np.asarray(y_true)

    def objective(candidate):
        candidate = np.sort(np.clip(candidate, 0.01, 0.99))
        preds = (probs > candidate.reshape(1, -1)).sum(axis=1)
        return -cohen_kappa_score(y_true, preds, weights="quadratic")

    result = minimize(
        objective,
        x0=np.full(NUM_CLASSES - 1, 0.5),
        method="Nelder-Mead",
        options={"maxiter": 200, "xatol": 1e-3, "fatol": 1e-4},
    )
    thresholds = np.sort(np.clip(result.x, 0.01, 0.99))
    return thresholds, result

def run_epoch(
    model,
    loader,
    optimizer=None,
    scheduler=None,
    desc="epoch",
    thresholds=None,
    return_logits=False,
 ):
    is_train = optimizer is not None
    model.train(is_train)

    all_true, all_pred, all_logits = [], [], []
    total_loss = 0.0

    if is_train:
        optimizer.zero_grad(set_to_none=True)

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        progress = tqdm(loader, desc=desc, leave=False)
        for batch in progress:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            y_idx = batch["label"].to(DEVICE)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = corn_loss(logits, y_idx, num_classes=NUM_CLASSES)

            if is_train:
                loss.backward()
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                if scheduler is not None:
                    scheduler.step()

            pred_idx = predict_from_logits(logits.detach(), thresholds=thresholds).cpu().numpy()
            true_idx = y_idx.detach().cpu().numpy()
            all_pred.extend(pred_idx.tolist())
            all_true.extend(true_idx.tolist())
            total_loss += loss.item() * input_ids.size(0)

            if return_logits:
                all_logits.append(logits.detach().cpu())

            avg_loss_so_far = total_loss / max(1, len(all_true))
            progress.set_postfix(loss=f"{avg_loss_so_far:.4f}")

    epoch_loss = total_loss / len(loader.dataset)
    qwk = cohen_kappa_score(all_true, all_pred, weights="quadratic")
    all_logits_tensor = torch.cat(all_logits, dim=0) if all_logits else torch.empty((0, NUM_CLASSES - 1))
    return epoch_loss, qwk, all_true, all_pred, all_logits_tensor

Device: cuda


In [6]:
# Build OOF logits for training (single export cell)
from sklearn.model_selection import GroupKFold

if "year" not in df.columns:
    raise KeyError("year column not found for year-wise folds")

year_groups = df.loc[data.index, "year"]
years_sorted = np.sort(year_groups.unique())
year_bins = np.array_split(years_sorted, 3)
year_bucket_map = {year: idx for idx, years in enumerate(year_bins) for year in years}
year_bucket = year_groups.map(year_bucket_map)
gkf = GroupKFold(n_splits=3)

OOF_EPOCHS = 1
RUN_OOF = True

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def logits_to_class_probs(logits_tensor: torch.Tensor) -> torch.Tensor:
    cond_probs = torch.sigmoid(logits_tensor)
    cum_probs = torch.cumprod(cond_probs, dim=1)
    probs = torch.zeros((cum_probs.size(0), NUM_CLASSES), dtype=cum_probs.dtype)
    probs[:, 0] = 1.0 - cum_probs[:, 0]
    for c in range(1, NUM_CLASSES - 1):
        probs[:, c] = cum_probs[:, c - 1] - cum_probs[:, c]
    probs[:, NUM_CLASSES - 1] = cum_probs[:, NUM_CLASSES - 2]
    probs = torch.clamp(probs, min=0.0)
    return probs / probs.sum(dim=1, keepdim=True).clamp(min=1e-12)

if RUN_OOF:
    oof_logits = torch.zeros((len(data), NUM_CLASSES - 1), dtype=torch.float32)
    oof_fold = np.full(len(data), -1)
    for fold, (train_idx, val_idx) in enumerate(gkf.split(data, data["target_triage_acuity"], groups=year_bucket), start=1):
        X_train = data.iloc[train_idx]["text_clean"].fillna("")
        y_train = data.iloc[train_idx]["target_triage_acuity"].astype(int)
        X_val = data.iloc[val_idx]["text_clean"].fillna("")
        y_val = data.iloc[val_idx]["target_triage_acuity"].astype(int)

        train_ds = OrdinalTextDataset(X_train, y_train, tokenizer, max_len=MAX_LEN)
        val_ds = OrdinalTextDataset(X_val, y_val, tokenizer, max_len=MAX_LEN)

        class_counts = y_train.value_counts().sort_index()
        class_count_tensor = torch.tensor(
            [class_counts.get(i, 1) for i in range(NUM_CLASSES)],
            dtype=torch.float32,
        )
        class_weights = 1.0 / torch.sqrt(class_count_tensor)
        sample_weights = class_weights[y_train.to_numpy(dtype=int)]
        train_sampler = WeightedRandomSampler(
            weights=sample_weights.detach().cpu().double().tolist(),
            num_samples=len(sample_weights),
            replacement=True,
        )

        train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, sampler=train_sampler, num_workers=0)
        val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0)

        model_fold = OrdinalBert(MODEL_NAME, num_classes=NUM_CLASSES).to(DEVICE)
        optimizer = torch.optim.AdamW(model_fold.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        total_steps = max(1, len(train_loader) * OOF_EPOCHS)
        warmup_steps = int(0.1 * total_steps)
        scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

        for epoch in range(1, OOF_EPOCHS + 1):
            _ = run_epoch(
                model_fold,
                train_loader,
                optimizer=optimizer,
                scheduler=scheduler,
                desc=f"OOF train f{fold} e{epoch}",
            )

        _, _, _, _, val_logits = run_epoch(
            model_fold,
            val_loader,
            desc=f"OOF val f{fold}",
            return_logits=True,
        )
        oof_logits[val_idx] = val_logits
        oof_fold[val_idx] = fold
        print(f"OOF fold {fold} done. Val rows: {len(val_idx)}")

    oof_probs = logits_to_class_probs(oof_logits)
    oof_df = pd.DataFrame({
        "row_index": data.index.to_numpy(),
        "year": year_groups.to_numpy(),
        "fold": oof_fold,
    })
    for k in range(NUM_CLASSES - 1):
        oof_df[f"raw_logit_t{k+1}"] = oof_logits[:, k].numpy()
    for c in range(NUM_CLASSES):
        oof_df[f"prob_class_{c+1}"] = oof_probs[:, c].numpy()

    oof_path = PROJECT_ROOT / "working_data" / "nlp_oof_logits_probs.csv"
    oof_path.parent.mkdir(parents=True, exist_ok=True)
    oof_df.to_csv(oof_path, index=False)
    print(f"Saved {len(oof_df):,} rows to: {oof_path}")
    display(oof_df.head())

/tmp/ipykernel_23976/2819323807.py:48: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  sample_weights = class_weights[y_train.to_numpy(dtype=int)]


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpie/distil-clinicalbert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


OOF train f1 e1:   0%|          | 0/948 [00:00<?, ?it/s]

OOF val f1:   0%|          | 0/1738 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: nlpie/distil-clinicalbert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


OOF train f3 e1:   0%|          | 0/1498 [00:00<?, ?it/s]

OOF val f3:   0%|          | 0/638 [00:00<?, ?it/s]

OOF fold 3 done. Val rows: 10207
Saved 58,124 rows to: /home/gaurav/python/kaggle/triage/working_data/nlp_oof_logits_probs.csv


,row_index,year,fold,raw_logit_t1,raw_logit_t2,prob_class_1,prob_class_2,prob_class_3
0,0,2018,1,3.198829,1.449911,0.039210,0.182565,0.778225
1,1,2018,1,2.578423,0.681613,0.070540,0.312207,0.617253
2,2,2018,1,2.424137,0.618163,0.081351,0.321710,0.596939
3,3,2018,1,1.241654,-2.707810,0.224148,0.727350,0.048502
4,4,2018,1,2.745491,1.066302,0.060342,0.240653,0.699005


In [9]:
# Save fine-tuned model artifacts for downstream explainability
import json
from datetime import datetime, timezone

MODEL_TAG = f"distilbert_corn_seed{SEED}"
artifact_dir = PROJECT_ROOT / "results" / "model_artifacts" / MODEL_TAG
tokenizer_dir = artifact_dir / "tokenizer"
weights_path = artifact_dir / "model_state.pt"
metadata_path = artifact_dir / "metadata.json"

artifact_dir.mkdir(parents=True, exist_ok=True)
tokenizer_dir.mkdir(parents=True, exist_ok=True)

torch.save(model_fold.state_dict(), weights_path)
tokenizer.save_pretrained(tokenizer_dir)

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "model_name": MODEL_NAME,
    "num_classes": int(NUM_CLASSES),
    "max_len": int(MAX_LEN),
    "dropout": 0.5,
    "seed": int(SEED),
    "weights_path": str(weights_path),
    "tokenizer_dir": str(tokenizer_dir),
    "class_labels": [0, 1, 2],
}

with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved model weights: {weights_path}")
print(f"Saved tokenizer dir: {tokenizer_dir}")
print(f"Saved metadata: {metadata_path}")
display(pd.DataFrame([metadata]))

Saved model weights: /home/gaurav/python/kaggle/triage/results/model_artifacts/distilbert_corn_seed42/model_state.pt
Saved tokenizer dir: /home/gaurav/python/kaggle/triage/results/model_artifacts/distilbert_corn_seed42/tokenizer
Saved metadata: /home/gaurav/python/kaggle/triage/results/model_artifacts/distilbert_corn_seed42/metadata.json


,created_at_utc,model_name,num_classes,max_len,dropout,seed,weights_path,tokenizer_dir,class_labels
0,2026-04-20T13:07:04+00:00,nlpie/distil-clinicalbert,3,100,0.5,42,/home/gaurav/python/kaggle/triage/results/mode...,/home/gaurav/python/kaggle/triage/results/mode...,"[0, 1, 2]"
